# Night of Code - Crypto Market Data Pipeline
### Fetch crypto data from CoinGecko API -> Load into PostgreSQL

---

**What we will build:**
```
CoinGecko API  →  Python (Extract & Transform)  →  PostgreSQL (Load)
```

**Requirements:**
- A PostgreSQL database

---

## Install Dependencies

In [5]:
# Install the PostgreSQL driver
!pip install psycopg2-binary --quiet

print("✅ Dependencies installed!")

✅ Dependencies installed!


'pip' is not recognized as an internal or external command,
operable program or batch file.


## Import Libraries

In [ ]:
import requests
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime
import pandas as pd

print(" Libraries imported!")

ModuleNotFoundError: No module named 'requests'

## Database Connection



In [ ]:
# ⚠️ UPDATE THESE WITH YOUR OWN CREDENTIALS
DB_CONFIG = {
    "host":     "<YOUR_AIVEN_HOST>",   # From Aiven dashboard
    "port":     <YOUR_PORT>,
    "dbname":   "<YOUR_DATABASE>",
    "user":     "<YOUR_USERNAME>",
    "password": "<YOUR_PASSWORD>",
    "sslmode":  "require"                 # Required for Aiven and most cloud DBs
}

def get_connection():
    """Create and return a PostgreSQL connection."""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        print("✅ Connected to PostgreSQL successfully!")
        return conn
    except Exception as e:
        print(f"❌ Connection failed: {e}")
        raise

# Test the connection
conn = get_connection()
conn.close()

## Create the Database Table

We'll run the SQL directly from Python 

In [ ]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS crypto_prices (
    id               SERIAL PRIMARY KEY,
    coin_id          VARCHAR(50)   NOT NULL,
    symbol           VARCHAR(10)   NOT NULL,
    name             VARCHAR(100)  NOT NULL,
    price_usd        NUMERIC(20,8) NOT NULL,
    market_cap       NUMERIC(30,2),
    total_volume     NUMERIC(30,2),
    price_change_24h NUMERIC(10,4),
    high_24h         NUMERIC(20,8),
    low_24h          NUMERIC(20,8),
    fetched_at       TIMESTAMP DEFAULT NOW()
);
"""

def create_table():
    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute(CREATE_TABLE_SQL)
        conn.commit()
        print("✅ Table 'crypto_prices' created (or already exists)!")
    except Exception as e:
        conn.rollback()
        print(f"❌ Failed to create table: {e}")
    finally:
        cursor.close()
        conn.close()

create_table()

## Extract: Fetch Data from CoinGecko API

In [ ]:
COINGECKO_URL = "https://api.coingecko.com/api/v3/coins/markets"

# Coins to track — add or remove any from this list
COINS_TO_TRACK = [
    "bitcoin", "ethereum", "solana", "binancecoin",
    "cardano", "ripple", "dogecoin", "polkadot"
]

def fetch_crypto_data(coin_ids):
    """Fetch current market data from CoinGecko."""
    params = {
        "vs_currency": "usd",
        "ids": ",".join(coin_ids),
        "order": "market_cap_desc",
        "per_page": 50,
        "page": 1,
        "sparkline": False
    }

    print(f"Fetching: {', '.join(coin_ids)}...")

    response = requests.get(COINGECKO_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    print(f"Fetched {len(data)} coins!")
    return data

# Run it and preview the raw response
raw_data = fetch_crypto_data(COINS_TO_TRACK)

# Show what the API returns (first coin only)
print("\n Sample raw API response (first coin):")
for key, value in list(raw_data[0].items())[:10]:
    print(f"  {key}: {value}")

## Transform: Clean & Shape the Data

In [ ]:
def transform_data(raw_data):
    """Extract only the fields we need and handle missing values."""
    records = []
    fetched_at = datetime.utcnow()

    for coin in raw_data:
        record = (
            coin.get("id"),
            coin.get("symbol", "").upper(),
            coin.get("name"),
            coin.get("current_price") or 0,
            coin.get("market_cap") or 0,
            coin.get("total_volume") or 0,
            coin.get("price_change_percentage_24h") or 0.0,
            coin.get("high_24h") or 0,
            coin.get("low_24h") or 0,
            fetched_at
        )
        records.append(record)

    print(f"🔄 Transformed {len(records)} records")
    return records

clean_records = transform_data(raw_data)

# Preview the cleaned data as a DataFrame (easier to read)
columns = ["coin_id", "symbol", "name", "price_usd", "market_cap",
           "total_volume", "price_change_24h", "high_24h", "low_24h", "fetched_at"]

df_preview = pd.DataFrame(clean_records, columns=columns)
print("\n Cleaned data preview:")
df_preview[["name", "symbol", "price_usd", "price_change_24h", "market_cap"]]

## Load: Insert Data into PostgreSQL

In [ ]:
INSERT_QUERY = """
    INSERT INTO crypto_prices (
        coin_id, symbol, name, price_usd, market_cap,
        total_volume, price_change_24h, high_24h, low_24h, fetched_at
    ) VALUES %s
"""

def load_to_postgres(records):
    """Batch insert all records into PostgreSQL."""
    conn = get_connection()
    cursor = conn.cursor()

    try:
        execute_values(cursor, INSERT_QUERY, records)
        conn.commit()   
        print(f" Inserted {len(records)} rows into crypto_prices")
    except Exception as e:
        conn.rollback() # Undo if something went wrong
        print(f" Insert failed: {e}")
        raise
    finally:
        cursor.close()
        conn.close()

load_to_postgres(clean_records)

## Run SQL Queries to Explore Your Data

Now let's query the database directly from Colab!

In [ ]:
def run_query(sql, columns):
    """Run any SQL query and return results as a DataFrame."""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(sql)
    rows = cursor.fetchall()
    cursor.close()
    conn.close()
    return pd.DataFrame(rows, columns=columns)

In [ ]:
# ── Query 1: All data, newest first ──
print(" All coins loaded:")
run_query("""
    SELECT name, symbol, price_usd, price_change_24h, fetched_at
    FROM crypto_prices
    ORDER BY fetched_at DESC, market_cap DESC;
""", ["Name", "Symbol", "Price (USD)", "24h Change (%)", "Fetched At"])

In [ ]:
# ── Query 2: Top 5 by market cap ──
print(" Top 5 coins by Market Cap:")
run_query("""
    SELECT name, symbol, price_usd, market_cap
    FROM crypto_prices
    ORDER BY market_cap DESC
    LIMIT 5;
""", ["Name", "Symbol", "Price (USD)", "Market Cap"])

In [ ]:
# ── Query 3: Biggest 24h gainers ──
print(" Biggest 24h Gainers:")
run_query("""
    SELECT name, symbol, price_usd, price_change_24h
    FROM crypto_prices
    ORDER BY price_change_24h DESC
    LIMIT 5;
""", ["Name", "Symbol", "Price (USD)", "24h Change (%)"])

In [ ]:
# ── Query 4: Biggest 24h losers ──
print(" Biggest 24h Losers:")
run_query("""
    SELECT name, symbol, price_usd, price_change_24h
    FROM crypto_prices
    ORDER BY price_change_24h ASC
    LIMIT 5;
""", ["Name", "Symbol", "Price (USD)", "24h Change (%)"])

In [ ]:
# ── Query 5: How many pipeline runs? ──
print(" Pipeline run count:")
run_query("""
    SELECT COUNT(DISTINCT fetched_at) AS total_runs,
           COUNT(*) AS total_rows
    FROM crypto_prices;
""", ["Total Runs", "Total Rows"])

## Run the Full Pipeline in One Cell

Run this cell multiple times to keep loading fresh data into your database!

In [ ]:
def run_pipeline():
    print("=" * 50)
    print("   Crypto ETL Pipeline — Night of Code")
    print("=" * 50)

    raw   = fetch_crypto_data(COINS_TO_TRACK)
    clean = transform_data(raw)
    load_to_postgres(clean)

    print("\n Done! Here's what's in your database now:")
    return run_query("""
        SELECT name, symbol, price_usd, price_change_24h, fetched_at
        FROM crypto_prices
        ORDER BY fetched_at DESC, market_cap DESC
        LIMIT 10;
    """, ["Name", "Symbol", "Price (USD)", "24h Change (%)", "Fetched At"])

run_pipeline()

In [ ]:
#  Write your own SQL query here!
run_query("""
    -- Your query here
    SELECT * FROM crypto_prices LIMIT 5;
""", ["id", "coin_id", "symbol", "name", "price_usd", "market_cap",
       "total_volume", "price_change_24h", "high_24h", "low_24h", "fetched_at"])